This notebook computes the accuracy of the LLM generated specifications for each of the prompting approaches

It loads in the specifcation generated LLMs in  run_llm.ipynb

You can run all cells in the notebook to compute and view the accuracy of all approaches using generated specification used in the paper's evaluation

You can also view the accuracy results in Plot_results.ipynb

To convenientialy run all cells: Select Run -> Run all cells option from the top of this notebook to view the plots

In [ ]:
#select structured NL intermediate representation for specifications

import os
os.environ["STRUCTNL_MODE"] = "fretish" #or "PSP"

In [ ]:
#select the benchmark

#dataset_list = ["Ventilator","RobotExplain","FSM-AP","FSM-S","REG","deepstl-test"]
#dataset_list = ["Thales"]
dataset_list = ["Ventilator"]

In [ ]:
#specify the model and number of candidates

num_trial = 10
model_list = ["gemini-2.5-flash"]
#model_list = ["gpt-4.1"]

In [ ]:
#select the current prompting method
num_trial = 10
model = "gemini-2.5-flash" #or "gpt-4.1"
cur_method = "nl2structnl" #ARTEMIS
#cur_method = "nl2ltl" #directTL
#cur_method = "nl2ltltemplate" #directTL-t
#cur_method = "synthtl" #SynthTL
#cur_method = "nl2spec" #nl2spec
#cur_method = "NL2TL-FT" #NL2TL+ (FT)
#cur_method = "deepstl" #DeepSTL (FT)

cur_list = [(cur_method,None)]

In [ ]:
import pandas as pd
import json
import numpy as np
import itertools
import os
os.environ["DATA_HOME_DIR"] = "./metadata"
os.environ["SMV_FILE_DIR"] = "./metadata/tmp"

In [ ]:
import data_loader
import nl2structnl
import nl2structnl_PSP
import nl2ltl
import metrics
import batch_check

In [ ]:
if os.getenv("STRUCTNL_MODE") == "fretish":
    result_dir = "./fretish_results/"
    data_home_dir = "./benchmarks/fret_specs/"
else:
    result_dir = "./PSP_results/"
    data_home_dir = "./benchmarks/psp_specs/"

In [ ]:
def get_max_num_trial_outputs(result_dir,cur_dataset_name,row_idx,model,cur_method,cur_mode,structnl):
    for num_trial in [50*(i+1) for i in range(20)][::-1]:
        try:
            cur_outputs, output_ltl_list = data_loader.load_outputs(result_dir,cur_dataset_name,row_idx,model,num_trial,cur_method,cur_mode,structnl=structnl)
            if num_trial != len(output_ltl_list):
                print(row_idx,len(output_ltl_list),num_trial)
            break
        except:
            pass
    return cur_outputs, output_ltl_list

In [ ]:
def is_need_run(in_file, res_file):
    assert os.path.exists(in_file)
    if not os.path.exists(res_file):
        return True  # No result yet
    mtime_input = os.path.getmtime(in_file)
    mtime_result = os.path.getmtime(res_file)
    return mtime_input >= mtime_result

In [ ]:
print(model_list)
print(dataset_list)
print(cur_list)

In [ ]:
missing_idx = []
for cur_dataset_name, model in itertools.product(dataset_list,model_list):
    cur_df_file = data_home_dir + cur_dataset_name + "/PlausibleSpecs.xlsx"
    df = pd.read_excel(cur_df_file, engine='openpyxl')
    row_idx_range = [i for i in range(0,len(df),1)]
    for row_idx in row_idx_range:
        #print(row_idx)
        for cur_method, cur_mode in cur_list:
            cur_exp_name = f"{result_dir}/{cur_dataset_name}-{row_idx}_model-{model}_trials-{num_trial}"
            if cur_mode == "extra":
                save_name = cur_mode + "_" + cur_method
            else:
                save_name = cur_method
            if not os.path.exists(cur_exp_name+"_" + cur_method + ".json"):
                continue
            if is_need_run(in_file=f"{result_dir}/{cur_dataset_name}-{row_idx}_model-{model}_trials-{num_trial}"+ "_" + cur_method +".json",
                           res_file=cur_exp_name+"_" + save_name + "_metrics.json"):
                print(cur_dataset_name,model,cur_method,row_idx)
                try:
                    label_output_list,label_ltl_list = data_loader.load_labels(data_home_dir,cur_dataset_name,row_idx,max_N_DURATION=1,structnl=os.getenv("STRUCTNL_MODE"))
                    cur_outputs, output_ltl_list = data_loader.load_outputs(result_dir,cur_dataset_name,row_idx,model,num_trial,cur_method,cur_mode,structnl=os.getenv("STRUCTNL_MODE"))
                    assert len(output_ltl_list) == num_trial, "missing output!"
                    assert len(output_ltl_list) > 0, "no outputs!"
                except Exception as e:
                    print(e)
                    #print(cur_dataset_name,cur_method,model,row_idx,'error load',e)
                    missing_idx.append((cur_dataset_name,model,row_idx,cur_method))
                    continue
                print("num outputs:",len(output_ltl_list))
                output_metrics = metrics.get_all_metrics_old(output_ltl_list,label_ltl_list,timeout=10,bmc_k=150,
                                                             #nusmv_jobs_per_thread=10
                                                            )
                
                #[equiv,subset,superset,subset&superset,overlap]
                metric_idx = 0
                print("num plausible:",np.sum(output_metrics,axis=0)[metric_idx])
                with open(cur_exp_name+"_" + save_name + "_metrics.json", "w") as json_file:
                    json.dump(output_metrics, json_file)
                

the following cells load all results in all benchmarks and computes the accuracy for the current method and model:

In [ ]:
num_trial = 10
model = "gemini-2.5-flash" #or "gpt-4.1"
cur_method = "nl2structnl" #ARTEMIS
#cur_method = "nl2ltl" #directTL
#cur_method = "nl2ltltemplate" #directTL-t
#cur_method = "synthtl" #SynthTL
#cur_method = "nl2spec" #nl2spec
#cur_method = "NL2TL-FT" #NL2TL+ (FT)
#cur_method = "deepstl" #DeepSTL (FT)

In [ ]:
fret_home_dir = "./benchmarks/fret_specs/"
psp_home_dir = "./benchmarks/psp_specs/"
fret_result_dir = "fretish_results"
psp_result_dir = "PSP_results"
cur_dataset_list = [
    (fret_home_dir,"Ventilator","fretish",fret_result_dir),
    (fret_home_dir,"RobotExplain","fretish",fret_result_dir),
    (fret_home_dir,"FSM-AP","fretish",fret_result_dir),
    (fret_home_dir,"FSM-S","fretish",fret_result_dir),
    (fret_home_dir,"REG","fretish",fret_result_dir),
    (fret_home_dir,"deepstl-test","fretish",fret_result_dir),
    (psp_home_dir,"Thales","PSP",psp_result_dir),
]
max_N_DURATION=1
cur_mode = None
is_correct = []
num_plausible_per_dataset = []
for home_dir,cur_dataset_name,structnl,result_dir in cur_dataset_list:
    cur_df_file = home_dir + cur_dataset_name + "/PlausibleSpecs.xlsx"
    df = pd.read_excel(cur_df_file, engine='openpyxl')
    row_idx_range = [i for i in range(0,len(df),1)]
    num_plausible = []
    for row_idx in row_idx_range:
        label_output_list,label_ltl_list = data_loader.load_labels(home_dir,cur_dataset_name,row_idx,structnl=structnl)
        num_plausible.append(len(label_output_list))
        cur_outputs, output_ltl_list = data_loader.load_outputs(result_dir,cur_dataset_name,row_idx,model,num_trial,cur_method,cur_mode,structnl=structnl)
        cur_exp_name = f"{result_dir}/{cur_dataset_name}-{row_idx}_model-{model}_trials-{len(cur_outputs)}"
        if cur_mode == "extra":
            save_name = cur_mode + "_" + cur_method
        else:
            save_name = cur_method
        output_metrics = json.load(open(cur_exp_name+"_" + save_name + "_metrics.json", "r"))
        if np.sum(output_metrics,axis=0)[0] == 0:
            is_correct.append(False)            
        else:
            is_correct.append(True)
    num_plausible_per_dataset.append(num_plausible)
print("accuracy:",np.mean(is_correct))